# 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from wordcloud import WordCloud
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer


#For Calculation of Performance of Models
from sklearn import metrics
from sklearn.metrics import f1_score,precision_score,recall_score,accuracy_score


#For Modelling Purpose
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.multiclass import OneVsRestClassifier

import time
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import nltk

nltk.download('wordnet')
from nltk.corpus import wordnet
!unzip /usr/share/nltk_data/corpora/wordnet.zip -d /usr/share/nltk_data/corpora/

In [ ]:
!pip install wordcloud

# 2. Loading the dataset

In [ ]:
books_df = pd.read_csv("/kaggle/input/books-dataset/BooksDataset.csv")

books_df.head()

In [ ]:
books_df.columns

# 3. Data Cleaning

### i. Removing unnecessary Columns

In [ ]:
books_df = books_df.drop(columns=['Price', 'Publish Date','Publisher','Authors'])

In [ ]:
books_df.head()

### ii. Handling Missing Values

In [ ]:
# Identifying missing values
print(books_df.isnull().sum())

In [ ]:
books_df.shape

In [ ]:
# Drop rows where either 'Description' or 'Category' is missing
df_cleaned = books_df.dropna(subset=['Description', 'Category'])
print(df_cleaned.shape)  # Check the number of remaining rows

In [ ]:
# Identifying missing values
print(df_cleaned.isnull().sum())

In [ ]:
# Save the cleaned DataFrame to a CSV file
df_cleaned.to_csv('cleaned_books_dataset03.csv', index=False)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.nunique()

### iii. Handling Duplicate Values

In [ ]:
duplicate_titles = df_cleaned[df_cleaned['Title'].duplicated(keep=False)]
num_duplicates = df_cleaned['Title'].duplicated().sum()
print(f"Number of duplicate titles: {num_duplicates}")

In [ ]:
df_cleaned = df_cleaned.drop_duplicates(subset=['Title'], keep='first')

In [ ]:
df_cleaned.nunique()

### iv. Text Preprocessing

In [ ]:
# Define stopwords
Stopwords = set(stopwords.words('english'))

# Initialize lemmatizer and stemmer
lemmatizer = WordNetLemmatizer()
stemmer = SnowballStemmer(language='english')

def clean_and_process(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    # Tokenization
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [word for word in tokens if word not in Stopwords]
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    # Reconstruct the text
    text = " ".join(tokens)
    # Remove words with length <= 3
    tokens = [word for word in text.split() if len(word) > 3]
    text = " ".join(tokens)
    # Remove single characters
    text = re.sub(r"\s+[a-zA-Z]\s+", ' ', text)
    # Remove HTML tags
    text = re.sub('<.*?>+', ' ', text)
    # Remove new line characters
    text = re.sub('\n', ' ', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    return text

In [ ]:
df_cleaned['Description'] = df_cleaned['Description'].apply(clean_and_process)
df_cleaned['Title'] = df_cleaned['Title'].apply(clean_and_process)

In [ ]:
df_cleaned

In [ ]:
# Keep only first category in category column
df_cleaned['Category'] = df_cleaned['Category'].apply(lambda x: x.split(',')[0].strip())

In [ ]:
df_cleaned.head()

# 4. EDA

In [ ]:
df_cleaned['Category'].value_counts()

In [ ]:

# plt.figure(figsize=(10, 6))

# # Creating a histogram using seaborn
# sns.histplot(df_cleaned['Category'], kde=False, bins=len(df_cleaned['Category'].unique()), color='blue')

# plt.title('Histogram of Genre Counts')
# plt.xlabel('Genres')
# plt.ylabel('Count')
# plt.xticks(rotation=90)  # Rotate x-axis labels for better readability

# plt.show()

### Histogram

In [ ]:
category_counts = df_cleaned['Category'].value_counts().reset_index()
category_counts.columns = ['Genre', 'Count']

# Select the top N categories (Here top 10)
top_n = 10
top_categories = category_counts.nlargest(top_n, 'Count')

# Set the size of the plot
plt.figure(figsize=(10, 6))

# Create a histogram using seaborn for the top N genres with different colors
sns.histplot(data=top_categories, x='Genre', weights='Count', hue='Genre', multiple='stack', palette='husl', bins=top_n)

# Customize the plot
plt.title(f'Histogram of Top {top_n} Genre Counts')
plt.xlabel('Genres')
plt.ylabel('Count')
plt.xticks(rotation=90)  # Rotate x-axis labels for better readability

# Show the plot
plt.show()

In [ ]:
category_name = 'Cooking'
category_descriptions = df_cleaned[df_cleaned['Category'] == category_name]['Description']
text = ' '.join(category_descriptions)
# Generate the word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')

plt.title(f'Word Cloud for {category_name}')
plt.show()

### Word Cloud

In [ ]:
# Get unique categories
categories = df_cleaned['Category'].unique()

# Setting up the plot for a single-column layout
plt.figure(figsize=(10, len(categories) * 5))

# Looping through each category to generate and plot the word cloud
for i, category in enumerate(categories):
    plt.subplot(len(categories), 1, i + 1)

    category_descriptions = df_cleaned[df_cleaned['Category'] == category]['Description']

    text = ' '.join(category_descriptions)

    # Generate the word cloud
    wordcloud = WordCloud(width=300, height=200, background_color='white').generate(text)

    # Plot the word cloud
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(category)

plt.tight_layout()
plt.show()

In [ ]:
#Converting all the categorical features of 'category' to numerical
df_cleaned['Encoded'] = LabelEncoder().fit_transform(df_cleaned['Category'])
df_cleaned.head()

# 5. Model Training and Evaluation

In [ ]:
X = CountVectorizer().fit_transform(df_cleaned['Description'])
y = df_cleaned['Encoded']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [ ]:
models=[
    MultinomialNB(),
    LogisticRegression(max_iter=1000, tol=1e-4),
    DummyClassifier(),
    KNeighborsClassifier()
]

### One vs Rest Classifer

In [ ]:
Name = []
Accuracy = []
Precision = []
F1_Score = []
Recall = []
Time_Taken = []
for model in models:
    name = type(model).__name__
    Name.append(name)
    model = OneVsRestClassifier(model)
    begin = time.time()
    model.fit(X_train,y_train)
    prediction = model.predict(X_test)
    end = time.time()
    Accuracy.append(accuracy_score(prediction,y_test))
    Precision.append(precision_score(prediction,y_test,average = 'macro'))
    Recall.append(recall_score(prediction,y_test,average = 'macro'))
    F1_Score.append(f1_score(prediction,y_test,average = 'macro'))
    Time_Taken.append(end-begin)
    print(name + ' Successfully Trained')

In [ ]:
Dict = {'Name':Name,'Accuracy':Accuracy,'Precision_score':Precision,'Recall_score':Recall,
        'F1_score':F1_Score,'Time Taken':Time_Taken}
model_df = pd.DataFrame(Dict)
model_df

### Modeling without using One vs Rest Classifer

In [ ]:
Name = []
Accuracy = []
Precision = []
F1_Score = []
Recall = []
Time_Taken = []
for model in models:
    name = type(model).__name__
    Name.append(name)
    begin = time.time()
    model.fit(X_train,y_train)
    prediction = model.predict(X_test)
    end = time.time()
    Accuracy.append(accuracy_score(prediction,y_test))
    Precision.append(precision_score(prediction,y_test,average = 'macro'))
    Recall.append(recall_score(prediction,y_test,average = 'macro'))
    F1_Score.append(f1_score(prediction,y_test,average = 'macro'))
    Time_Taken.append(end-begin)
    print(name + ' Successfully Trained')

In [ ]:
Dict = {'Name':Name,'Accuracy':Accuracy,'Precision_score':Precision,'Recall_score':Precision,
        'F1_score':F1_Score,'Time Taken':Time_Taken}
model_df = pd.DataFrame(Dict)
model_df

In [ ]:
import plotly.express as px

In [ ]:
model_df.sort_values(by = 'Accuracy',ascending = False,inplace = True)
fig = px.line(model_df, x="Name", y="Accuracy", title='Accuracy VS Model')
fig.show()

In [ ]:
model_df.sort_values(by = 'Time Taken',ascending = False,inplace = True)
fig = px.line(model_df, x="Name", y="Time Taken", title='Time Taken VS Model')
fig.show()

### Conclusion

The Multinomial NB takes less time.The Logistic Regression has the highest Accuracy.